# ANGSI JR. V10.2 — JupyterLab launch

Full instructions: **README_V10_2.md**. Execute each cell top-to-bottom. If installed packages require kernel restart, restart then rerun from Cell 1.


In [ ]:
from pathlib import Path
import zipfile
ROOT=Path('/data/epic20/APGCE/apgce_team_13')
ZIP=ROOT/'ANGSI_JR_V10_2_FULL.zip'
if ZIP.is_file():
    with zipfile.ZipFile(ZIP) as z:
        assert z.testzip() is None
        z.extractall(ROOT)
APP_DIR=ROOT/'ANGSI_JR_V10_2_AUTO_VOLUMETRICS'
APP=APP_DIR/'angsi_jr_v10.py'
print('APP FOUND:', APP.exists())
assert APP.exists(),f'Upload and extract ZIP first: {ZIP}'


In [ ]:
%pip install -r /data/epic20/APGCE/apgce_team_13/ANGSI_JR_V10_2_AUTO_VOLUMETRICS/requirements.txt


In [ ]:
import os,sys,subprocess,py_compile,socket
from pathlib import Path
APP_DIR=Path('/data/epic20/APGCE/apgce_team_13/ANGSI_JR_V10_2_AUTO_VOLUMETRICS')
APP=APP_DIR/'angsi_jr_v10.py'
LOG=APP_DIR/'server_v102.log'
py_compile.compile(str(APP),doraise=True)
env=os.environ.copy()
env['ANGSI_DATA_ROOT']='/data/epic20/APGCE/input_data/challenge3'
with LOG.open('a') as out:
    proc=subprocess.Popen([sys.executable,'-m','streamlit','run',str(APP),
        '--server.address=127.0.0.1','--server.port=8519','--server.headless=true',
        '--server.fileWatcherType=none'],cwd=str(APP_DIR),stdout=out,stderr=subprocess.STDOUT,env=env,start_new_session=True)
print('PID',proc.pid,'Jupyter host',socket.gethostname(),'log',LOG)


In [ ]:
import urllib.request,time
healthy=False
for i in range(12):
    try:
        with urllib.request.urlopen('http://127.0.0.1:8519/_stcore/health',timeout=4) as r:
            print('HTTP',r.status,'body',r.read().decode());healthy=r.status==200;break
    except Exception as e:
        if i==11:print('Health error:',e)
        else:time.sleep(1)
if not healthy:print(LOG.read_text(errors='replace')[-8000:] if LOG.exists() else 'NO LOG')


## Open dashboard

On Firefox running on **the same HPC host as Streamlit**, open: `http://127.0.0.1:8519/`.

Other host? Try JupyterHub origin + `/user/apgce_team_13/proxy/8519/` **only when the proxy is enabled**; otherwise ask HPC admin or use supported port forwarding. Health HTTP 200 is not a browser connectivity guarantee.


In [ ]:
import socket,subprocess
print('HOST:',socket.gethostname())
p=subprocess.run(['ss','-ltn'],capture_output=True,text=True)
print('LISTENER:', '\n'.join(x for x in p.stdout.splitlines() if ':8519' in x) or 'none')
print('RECENT LOG:',LOG.read_text(errors='replace')[-8000:] if LOG.exists() else 'none')
